# 台灣好聲音（PitchLab）雲端 AI 分軌與歌詞生成器
### 操作只要 2 步（支援一次多首批次處理，精準上傳至 `麻糬紅豆湯/PitchLab_Songs`）：
1. 上方選單 **「執行階段」→「變更執行階段類型」→ 選 `T4 GPU`**。
2. 點擊下方 **「步驟 1」左側的 ▶ 播放鈕**：完成授權與安裝後，下方會**自動出現「選擇音樂檔」按鈕（可一次選多首）**，按下「開始 AI 分軌並上傳雲端」就會依序處理並上傳至 `PitchLab_Songs`，打開網頁 App 就會自動匯入！

In [ ]:
#@title 步驟 1：授權雲端硬碟、安裝環境並顯示批次上傳面板（點左側 ▶ 執行）
from google.colab import auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload
from pathlib import Path
import subprocess
import ipywidgets as widgets
from IPython.display import display, clear_output

auth.authenticate_user()
drive_svc = build('drive', 'v3')

PARENT_FOLDER_ID = '1dO5Gvb1naqbodVf0-_n0GEm62Akk-12P'  # 麻糬紅豆湯
TARGET_SUBFOLDER = 'PitchLab_Songs'

def get_or_create_target_folder():
    q = f"'{PARENT_FOLDER_ID}' in parents and name='{TARGET_SUBFOLDER}' and mimeType='application/vnd.google-apps.folder' and trashed=false"
    res = drive_svc.files().list(q=q, fields='files(id, name)').execute()
    items = res.get('files', [])
    if items:
        return items[0]['id']
    meta = {'name': TARGET_SUBFOLDER, 'mimeType': 'application/vnd.google-apps.folder', 'parents': [PARENT_FOLDER_ID]}
    created = drive_svc.files().create(body=meta, fields='id').execute()
    return created['id']

def upload_zip_to_drive(zip_path: Path, folder_id: str):
    media = MediaFileUpload(str(zip_path), mimetype='application/zip', resumable=True)
    safe_name = zip_path.name.replace("'", "\\'")
    q = f"'{folder_id}' in parents and name='{safe_name}' and trashed=false"
    existing = drive_svc.files().list(q=q, fields='files(id, name)').execute().get('files', [])
    if existing:
        return drive_svc.files().update(fileId=existing[0]['id'], media_body=media, fields='id, name').execute()
    meta = {'name': zip_path.name, 'parents': [folder_id]}
    return drive_svc.files().create(body=meta, media_body=media, fields='id, name').execute()

target_folder_id = get_or_create_target_folder()

!git clone https://github.com/bard0228/pitchlab.git /content/pitchlab 2>/dev/null || (cd /content/pitchlab && git pull)
!pip -q install -r /content/pitchlab/pipeline/requirements.txt
!apt-get -qq install -y ffmpeg

uploader = widgets.FileUpload(accept='.mp3,.wav,.m4a,.flac,.ogg,.aac', multiple=True, description='選擇音樂檔(可多選)')
title_box = widgets.Text(value='', placeholder='單首可自訂；多首自動使用各檔名', description='歌名：')
lyrics_dd = widgets.Dropdown(
    options=[('自動辨識歌詞 (Whisper)', 'whisper'), ('不產生歌詞', 'none')],
    value='whisper',
    description='歌詞：',
)
run_btn = widgets.Button(description='開始 AI 分軌並上傳雲端', button_style='success', icon='play')
status_out = widgets.Output()

def _extract_uploaded_all(val):
    items = []
    if isinstance(val, dict) and val:
        for name, info in val.items():
            items.append((name, bytes(info['content'])))
    elif isinstance(val, (tuple, list)):
        for info in val:
            items.append((info['name'], bytes(info['content'])))
    return items

def _on_run(_):
    with status_out:
        clear_output()
        files_list = _extract_uploaded_all(uploader.value)
        if not files_list:
            print('⚠️ 請先點擊上方「選擇音樂檔」上傳 MP3 / WAV / M4A（可一次多選）！')
            return
        in_dir = Path('/content/pitchlab_in')
        out_dir = Path('/content/pitchlab_out')
        in_dir.mkdir(exist_ok=True)
        out_dir.mkdir(exist_ok=True)

        total = len(files_list)
        ok_count = 0
        fail_names = []
        print(f'📦 共收到 {total} 首歌，開始依序處理與上傳...\n')

        for idx, (fname, fbytes) in enumerate(files_list, 1):
            src_path = in_dir / fname
            src_path.write_bytes(fbytes)
            song_title = title_box.value.strip() if (total == 1 and title_box.value.strip()) else src_path.stem
            out_zip = out_dir / f'{song_title}.zip'
            cmd = [
                'python3', '/content/pitchlab/pipeline/process_song.py',
                '--input', str(src_path),
                '--separator', 'demucs',
                '--title', song_title,
                '--out', str(out_zip),
            ]
            if lyrics_dd.value == 'whisper':
                cmd += ['--lyrics-mode', 'whisper']
            print(f'[{idx}/{total}] 🎵 正在處理《{song_title}》（Demucs 分軌 + {lyrics_dd.value}）...')
            res = subprocess.run(cmd, check=False)
            if res.returncode == 0 and out_zip.exists():
                print(f'[{idx}/{total}] ☁️ 正在上傳至 PitchLab_Songs...')
                uploaded_f = upload_zip_to_drive(out_zip, target_folder_id)
                print(f'[{idx}/{total}] ✅ 完成：{uploaded_f["name"]}\n')
                ok_count += 1
            else:
                print(f'[{idx}/{total}] ❌ 處理失敗：《{song_title}》\n')
                fail_names.append(song_title)

        print(f'🎉 批次作業結束！成功 {ok_count}/{total} 首' + (f'，失敗：{", ".join(fail_names)}' if fail_names else ''))
        if ok_count > 0:
            print('📱 打開手機「台灣好聲音」App（或點歌庫的「同步」）就會自動匯入全部新歌！')

run_btn.on_click(_on_run)
print(f'✅ 環境與雲端硬碟就緒（目標資料夾 ID：{target_folder_id}）！')
display(widgets.VBox([uploader, title_box, lyrics_dd, run_btn, status_out]))

In [ ]:
#@title 步驟 2（進階選用）：貼 YouTube 網址、純文字歌詞對齊或上傳 .lrc
YOUTUBE_URL = "" #@param {type:"string"}
SONG_TITLE = "" #@param {type:"string"}
LYRICS_MODE = "whisper" #@param ["whisper", "align_text", "upload_lrc", "none"]
PLAIN_LYRICS = "" #@param {type:"string"}

from google.colab import files
from pathlib import Path
import subprocess

tmp_dir = Path('/content/pitchlab_tmp')
tmp_dir.mkdir(exist_ok=True)

cmd = ['python3', '/content/pitchlab/pipeline/process_song.py', '--separator', 'demucs']
if YOUTUBE_URL.strip():
    cmd += ['--youtube', YOUTUBE_URL.strip()]
else:
    print('👇 請點下方「選擇檔案」按鈕上傳 MP3 / WAV / M4A：')
    uploaded = files.upload()
    src_file = next(iter(uploaded.keys()))
    cmd += ['--input', src_file]
    if not SONG_TITLE.strip():
        SONG_TITLE = Path(src_file).stem

if LYRICS_MODE == 'upload_lrc':
    print('👇 請上傳對應的 .lrc 動態歌詞檔：')
    lrc_up = files.upload()
    lrc_file = next(iter(lrc_up.keys()))
    cmd += ['--lrc', lrc_file]
elif LYRICS_MODE == 'align_text':
    txt_path = tmp_dir / 'plain_lyrics.txt'
    if PLAIN_LYRICS.strip():
        txt_path.write_text(PLAIN_LYRICS.replace('\\n', '\n'), encoding='utf-8')
    else:
        print('👇 請上傳純文字歌詞 .txt 檔：')
        txt_up = files.upload()
        txt_path = Path(next(iter(txt_up.keys())))
    cmd += ['--lyrics-mode', 'align', '--lyrics-text', str(txt_path)]
elif LYRICS_MODE == 'whisper':
    cmd += ['--lyrics-mode', 'whisper']

if SONG_TITLE.strip():
    cmd += ['--title', SONG_TITLE.strip()]

out_name = (SONG_TITLE.strip() or 'song_package') + '.zip'
out_zip = tmp_dir / out_name
cmd += ['--out', str(out_zip)]

print('🎵 AI 分軌與歌詞處理中，請稍候...')
res = subprocess.run(cmd, check=False)
if res.returncode == 0 and out_zip.exists():
    uploaded_f = upload_zip_to_drive(out_zip, target_folder_id)
    print('✅ 完成！已上傳至雲端硬碟：', uploaded_f['name'])
else:
    print('❌ 處理失敗，請查看上方錯誤訊息（若 YouTube 被擋請留空 YOUTUBE_URL 改用上傳本機音檔）。')